# Activity: Estimating a Logistic Regression Binary Classifier using Gradient Descent
In this activity, we will implement a logistic regression binary classifier using gradient descent. Logistic regression is a statistical method for predicting binary outcomes based on one or more predictor variables. It is widely used in various fields, including machine learning, medical research, and social sciences.

> __Learning Objectives:__
>
> After completing this activity, students will be able to: 
> * __Derive logistic regression from the Boltzmann distribution:__ Develop the logistic regression model by viewing binary class labels as states in a Boltzmann distribution and derive the cross-entropy loss function.
> * __Implement gradient descent optimization:__ Use gradient descent to minimize the cross-entropy loss function and find optimal classifier parameters using finite difference gradient approximations.
> * __Train and evaluate a binary classifier:__ Train a logistic regression model on the banknote authentication dataset and evaluate its performance using a confusion matrix.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
The dataset we will explore is the [banknote authentication dataset from the UCI archive](https://archive.ics.uci.edu/dataset/267/banknote+authentication). This dataset has `1372` instances with 4 continuous features and an integer $\{-1,1\}$ class variable. 

> __Description of the dataset__ 
> 
> * Data were extracted from images taken from genuine and forged banknote-like specimens. An industrial camera, usually used for print inspection, was used for digitization. The final images have 400x400 pixels. Due to the object lens and distance to the investigated object, gray-scale pictures with a resolution of about 660 dpi were obtained. Wavelet Transform tools were used to extract features from images.
> * __Features__: The data has four continuous features from each image: `variance` of the wavelet transformed image, `skewness` of the wavelet transformed image, `kurtosis` of the wavelet transformed image, and the `entropy` of the wavelet transformed image. The class is $\{-1,1\}$ where a class value of `-1` indicates genuine and `1` indicates forged.

We've included this dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) and have provided [the `MyBanknoteAuthenticationDataset(...)` helper function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyBanknoteAuthenticationDataset) for easy access. 

This method returns the data in [a `DataFrame` instance](https://github.com/JuliaData/DataFrames.jl), which we'll save in the `df_banknote` variable.

In [2]:
df_banknote =  MyBanknoteAuthenticationDataset();

In [12]:
df_banknote

Row,variance,skewness,curtosis,entropy,class
,Float64,Float64,Float64,Float64,Int64
1,3.6216,8.6661,-2.8073,-0.44699,-1
2,4.5459,8.1674,-2.4586,-1.4621,-1
3,3.866,-2.6383,1.9242,0.10645,-1
4,3.4566,9.5228,-4.0112,-3.5944,-1
5,0.32924,-4.4552,4.5718,-0.9888,-1
6,4.3684,9.6718,-3.9606,-3.1625,-1
7,3.5912,3.0129,0.72888,0.56421,-1
8,2.0922,-6.81,8.4636,-0.60216,-1
9,3.2032,5.7588,-0.75345,-0.61251,-1


Now let's split the dataset into the system input matrix $\mathbf{X}$ (independent variables, characteristics of the banknote) and the output vector $\mathbf{y}$ (dependent variable, the banknote class).

The input matrix $\mathbf{X}$ will contain all the columns except for the `class` column (the output variable). The output vector $\mathbf{y}$ will contain only the `class` column.

In [3]:
X = Matrix(df_banknote[:, Not(:class)]); # data matrix: select all the columns *except* class
y = Vector(df_banknote[:, :class]); # output vector: select the class column

__Feature scaling__: Scaling the features to have zero mean and unit variance is important for regularization, as it ensures that all features contribute equally to the regularization penalty. Without scaling, features with larger magnitudes could dominate the regularization term, leading to suboptimal model performance.

In [4]:
X̂ = let
    
    # compute the mean and standard deviation of each feature (column)
    mu = mean(X, dims=1);
    sigma = std(X, dims=1);
    X̂ = (X .- mu) ./ sigma
end;

__Verify__ that the data is zero-mean and unit-variance after scaling.

In [5]:
mean(X̂, dims=1), std(X̂, dims=1)

([4.143106330962683e-17 0.0 1.0357765827406708e-17 -6.085187423601441e-17], [1.0 1.0 1.0 1.0])

Finally, let's partition the data into a `training` and `testing` set so that we can determine how well the model can predict unseen data, i.e., how well the model generalizes.

In [6]:
training, testing = let

    # initialize -
    s = 0.80; # fraction of data for training
    D = X̂; # use scaled features
    number_of_training_samples = Int(round(s * size(D,1))); # 80% of the data for training
    i = randperm(size(D,1)); # random permutation of the indices
    training_indices = i[1:number_of_training_samples]; # first 80% of the indices
    testing_indices = i[number_of_training_samples+1:end]; # last 20% of the indices    
    
    # setup training -
    one_vector = ones(number_of_training_samples);
    training = (X=[D[training_indices, :] one_vector], y=y[training_indices]);

    # setup testing -
    one_vector = ones(length(testing_indices));
    testing = (X=[D[testing_indices, :] one_vector], y=y[testing_indices]);
    training, testing;
end;

___

## Simple Gradient Descent Algorithm
Let's develop a simple gradient descent algorithm to minimize the unconstrained logistic regression problem where we need to minimize the negative log-likelihood, i.e., the cross-entropy loss function $J(\theta)$.

Gradient descent minimizes a function by iteratively adjusting the parameters in the _opposite direction_ of the gradient. 

> __Gradient?__
> 
> The gradient is a vector that points in the direction of the steepest increase of the function, and its magnitude indicates how steep the slope is in that direction. Thus, the negative gradient points in the direction of the steepest decrease of the function. We use the negative gradient to update the parameters in a way that reduces the value of the loss function, guiding us towards the minimum.

__Initialization:__ We start with an initial guess for the parameters $\theta^{(0)}\in\mathbb{R}^{p}$, which can be a random vector or a vector of zeros. Specify a learning rate $\alpha(k)>0$ (which can be constant or a function of the iteration count $k$) and a stopping criterion, e.g., a small threshold $\epsilon>0$ and a maximum number of iterations `maxiter`. Set $\texttt{converged}\gets\texttt{false}$ and the iteration count $k\gets0$.

While not $\texttt{converged}$ __do__:
1. Compute the update direction $\mathbf{d} = -\nabla_{\theta}L(\theta^{(k)})$, where $\nabla_{\theta}L(\theta^{(k)})$ is the gradient of the loss function with respect to the parameters at iteration $k$.
2. Update the parameters $\theta^{(k+1)} = \theta^{(k)} + \alpha(k)\cdot\mathbf{d}$.
3. Check the stopping criterion:
   - If $\lVert\theta^{(k+1)} - \theta^{(k)}\rVert\leq\epsilon$, set $\texttt{converged} = \texttt{true}$. Alternatively, if the gradient is small at the current iteration, i.e., $\lVert\nabla_{\theta}L(\theta^{(k)})\rVert\leq\epsilon$, set $\texttt{converged} = \texttt{true}$.
   - If $k\geq\texttt{maxiter}$, set $\texttt{converged} = \texttt{true}$.
     > __Optional__:
     >  Warn the user that the maximum number of iterations has been reached _without convergence_.
   - If neither of the above conditions is met, continue iterating.
4. Increment the iteration count $k\gets k+1$, and update the learning rate $\alpha(k)$, if applicable.

___

## Training and Testing the Logistic Regression Classifier
We implemented [the `MyLogisticRegressionClassificationModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/), which contains data required to solve the logistic regression problem, i.e., parameters, the learning rate, a stopping tolerance parameter $\epsilon$, and a loss (objective) function that we want to minimize. 

> __Details__
> 
> * __Technical note__: In this implementation, we approximated the gradient calculation using [a forward finite difference](https://en.wikipedia.org/wiki/Finite_difference). In general, this is not a great idea. This is one of my super pet peeves of gradient descent; computing the gradient is usually a hassle, and we do a bunch of function evaluations to get a good approximation of the gradient. However, finite difference is easy to implement.
> * __Note on the loss function__: In the code below, we use the natural logarithm `log` in the loss function. You could also use `log10`, which wouldn't change the location of the minimum since `log10` is simply a scaled version of the natural log ($\log_{10}(x) = \ln(x)/\ln(10)$). However, switching to `log10` _would_ scale the gradient magnitudes by $1/\ln(10)\approx 0.434$, so you would need to adjust the learning rate accordingly.
> * In the code block below, we [build a `model::MyLogisticRegressionClassificationModel` instance using a `build(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/). The model instance initially has a random guess for the classifier parameters. We use gradient descent to refine that guess [using the `learn(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/), which returns an updated model instance (with the best parameters that we found so far). 

We return the updated model instance and save it in the `model_logistic::MyLogisticRegressionClassificationModel` variable.

In [13]:
model_logistic = let

    # data -
    X = training.X; # feature matrix
    y = training.y; # labels
    number_of_features = size(X,2); # number of features + 1
    T = 1.0; # inverse temperature for logistic regression
    h = 1e-6; # step size for finite difference gradient approximation
    λ = 0.0; # regularization parameter
    ϵ = 1e-6; # tolerance for convergence
    maxiter = 20000; # maximum number of iterations for gradient descent

    # model
    model = build(MyLogisticRegressionClassificationModel, (
        parameters = 0.01*ones(number_of_features), # initial value for the parameters: these will be updated
        learning_rate = 0.005, # you pick this
        ϵ = ϵ, # you pick this (this is the tolerance for convergence)
        h = h, # you pick this (this is the step size for finite difference gradient approximation)
        λ = λ, # regularization parameter
        T = T, # inverse temperature for logistic regression
        loss_function = (x,y,T,λ,θ) -> log(1+exp(-2*y*T*(dot(x,θ)))) + λ*norm(θ,2)^2 # cross-entropy loss with L2 regularization
    ));

    # train -
    model = learn(X,y,model, maxiter = maxiter, verbose = true); # this is learning the model parameters

    # return -
    model;
end;

Stopped after number of iterations: 20001. We have error: 0.0031115266681660697


Let's use the updated `model_logistic::MyLogisticRegressionClassificationModel` instance (that has learned some parameters from the `training` data) and test how well we classify data that we have never seen, i.e., how well we classify the `test` dataset.

> __Inference__ 
>
> We run the classification operation on the (unseen) test data [using the `classify(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/). This method takes a feature array `X` and the (trained) model instance. It returns the probability of a label in the `P::Array{Float64,2}` array (which is different than the Perceptron). Each row of `P` corresponds to a test instance, in which each column corresponds to a label, in the case `1` and `-1`.

We store the actual (correct) label in the `y_banknote_logistic::Array{Int64,1}` vector. We compute the predicted label for each test instance by finding the highest probability column. We store the predicted labels in the `ŷ_banknote_logistic::Array{Int64,1}` vector.

In [14]:
ŷ_banknote_logistic,y_banknote_logistic, P = let

    # initialize -
    X = testing.X; # feature matrix
    y = testing.y; # labels
    number_of_examples = size(X,1); # how many examples do we have (rows)

    # compute the estimated labels -
    P = classify(X, model_logistic) # logistic regression returns a x x 2 array holding the probability

    # convert the probability to a choice ... for each row (test instance), compute the col with the highest probability
    ŷ = zeros(number_of_examples);
    for i ∈ 1:number_of_examples
        a = argmax(P[i,:]); # col index with largest value
        ŷ[i] = 1; # default
        if (a == 2)
            ŷ[i] = -1;
        end
    end
    
    # return -
    ŷ, y, P
end;

> __Error analysis__
>
> Total mistakes (or mistake percentage) is only part of the story. We should understand whether we are biased toward false positives or false negatives. How many times did we predict a banknote was forged when it was genuine (false positive)? How many times did we predict genuine when it was forged (false negative)? For this we use a confusion matrix. 
>
> The confusion matrix for a binary classifier is typically structured as:
>
>|                     | **Predicted Positive** | **Predicted Negative** |
>|---------------------|------------------------|------------------------|
>| **Actual Positive** | True Positive (TP)     | False Negative (FN)    |
>| **Actual Negative** | False Positive (FP)    | True Negative (TN)     |
> 
> From the confusion matrix, we derive three key metrics:
>
> * __Accuracy__ is the fraction of correct predictions overall: $\texttt{accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$. It tells us the overall success rate but can be misleading if classes are imbalanced, i.e., a classifier that predicts __negative__ for everything might achieve 95% accuracy on an imbalanced dataset.
>
> * __Precision__ answers: "When we predict positive, how often are we right?" $\texttt{precision} = \frac{TP}{TP + FP}$. In a fraud detection context, high precision means fewer false alarms, i.e., if you say a banknote is forged, it probably is.
>
> * __Recall__ (also called sensitivity) answers: "Of all the true positives, how many did we catch?" $\texttt{recall} = \frac{TP}{TP + FN}$. High recall means we're not missing forged banknotes. In fraud detection, recall is often more critical than precision because missing a forged banknote is worse than a false alarm.

We compute the __confusion matrix__ to get these counts. We use the [confusion(...) method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.confusion), which takes actual labels and estimated labels, returning the confusion matrix.

In [15]:
CM_logistic = confusion(y_banknote_logistic, ŷ_banknote_logistic)

2×2 Matrix{Int64}:
 113    1
   1  159

Let's compute the accuracy, precision, and recall for our logistic regression classifier on the test dataset. We store these metrics in the `accuracy_logistic`, `precision_logistic`, and `recall_logistic` variables, respectively.

In [16]:
accuracy_logistic, precision_logistic, recall_logistic = let

    # extract confusion matrix values -
    TP = CM_logistic[1,1]; # true positives
    TN = CM_logistic[2,2]; # true negatives
    FP = CM_logistic[2,1]; # false positives
    FN = CM_logistic[1,2]; # false negatives

    # compute metrics -
    accuracy = (TP + TN) / (TP + TN + FP + FN); # accuracy
    precision = TP / (TP + FP); # precision
    recall = TP / (TP + FN); # recall

    # print so the user can see -
    println("Logistic Regression Accuracy: $(round(accuracy*100,digits=2))%")
    println("Logistic Regression Precision: $(round(precision*100,digits=2))%")
    println("Logistic Regression Recall: $(round(recall*100,digits=2))%")

    # return -
    accuracy, precision, recall
end;

Logistic Regression Accuracy: 99.27%
Logistic Regression Precision: 99.12%
Logistic Regression Recall: 99.12%


What does the probability output look like? The first column corresponds to the probability of class `1` (forged), while the second column corresponds to the probability of class `-1` (genuine, not forged).

> __What do we expect to see?__ The probabilities should be close to `0` or `1` for most instances, indicating high confidence in the predictions. However, some instances may have probabilities closer to `0.5`, indicating uncertainty in the classification. As we change the inverse temperature parameter $T$, we expect the probabilities to become more or less extreme. 
> * As (inverse) $T\rightarrow{0}$ decreases, the system becomes hotter reflecting lower confidence in the predictions, i.e., more confusion. In this case, we expect more probabilities to be closer to `0.5`.
> * As (inverse) $T\rightarrow{\infty}$ increases, the system becomes colder and we expect the probabilities to be more extreme (closer to `0` or `1`), indicating higher confidence.

What do we see? (each row corresponds to a test instance, while each column corresponds to the probability of a class label):

In [17]:
P

274×2 Matrix{Float64}:
 0.999212     0.000787678
 1.54156e-15  1.0
 1.0          2.73978e-8
 1.84652e-5   0.999982
 1.0          2.63912e-11
 5.42995e-17  1.0
 1.0          3.04555e-12
 4.24888e-14  1.0
 1.68108e-15  1.0
 1.0          2.22755e-7
 ⋮            
 1.0          7.87862e-13
 4.51987e-14  1.0
 1.28225e-15  1.0
 1.0          8.48496e-10
 1.0          1.00465e-7
 1.0          1.50388e-10
 2.26468e-20  1.0
 0.985995     0.0140054
 2.50741e-18  1.0

___

## Summary
In this activity, we built a logistic regression binary classifier from first principles using gradient descent optimization. We derived the cross-entropy loss function from a Boltzmann distribution, implemented the gradient descent algorithm with finite difference gradient approximations, and trained the model on the banknote authentication dataset.

> __Key Takeaways:__
> 
> * **Cross-entropy loss minimization:** We minimized the negative log-likelihood function (cross-entropy loss) to find optimal classifier parameters. The gradient descent algorithm iteratively updated parameters using an appropriate learning rate until convergence.
> * **Training and testing data split:** We randomly partitioned the banknote dataset into training and testing subsets. This split allowed us to evaluate how well the trained model generalizes to unseen examples.
> * **Performance evaluation with confusion matrix:** We computed a confusion matrix to assess the classifier's performance on the test set. The matrix shows true positives, true negatives, false positives, and false negatives for genuine versus forged banknote classifications.

The logistic regression model successfully learned to classify banknotes based on wavelet-transformed image features, demonstrating the effectiveness of gradient descent for parameter estimation in binary classification problems.
___